<a href="https://colab.research.google.com/github/Amina-Mami/Web-Scraping-Books/blob/main/Books_Scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Web Scraping avec BeautifulSoup
## Extraire des livres depuis une boutique en ligne

---

**Objectif :** Scraper le site [books.toscrape.com](https://books.toscrape.com) pour extraire le **titre**, le **prix**, la **note** et la **disponibilité** de chaque livre.

**Site cible :** `https://books.toscrape.com` *(site fictif conçu pour s'entraîner au scraping)*


## Étape 0 — Installation des bibliothèques

In [ ]:
!pip install requests beautifulsoup4 lxml pandas --quiet
print(" Bibliothèques prêtes !")

 Bibliothèques prêtes !


## Étape 1 — Importer les bibliothèques
Importez `requests`, `BeautifulSoup` (depuis `bs4`) et `pandas`.

In [ ]:

import requests
from bs4 import BeautifulSoup
import pandas as pd

print("Imports OK")

Imports OK


## Étape 2 — Télécharger la page
Utilisez `requests.get()` pour récupérer la page.  
Vérifiez que le code de statut est **200** (succès).

In [ ]:
url = "https://books.toscrape.com"


response = requests.get(url, timeout=10)

print(f"Code de statut : {response.status_code}")


Code de statut : 200


## Étape 3 — Parser le HTML avec BeautifulSoup
Créez un objet `soup` à partir de `response.text`.  
Utilisez le parser `'lxml'`.

In [ ]:

soup = BeautifulSoup(response.text, "lxml")


print(f"Titre : {soup.title.text}")


Titre : 
    All products | Books to Scrape - Sandbox



## Étape 4 — Trouver tous les livres
Chaque livre est dans une balise `<article class="product_pod">`.  
Utilisez `soup.find_all()` pour les récupérer tous.

In [ ]:


articles = soup.find_all("article", class_="product_pod")

print(f"📚 Livres trouvés : {len(articles)}")


📚 Livres trouvés : 20


## Étape 5 — Préparer la conversion des étoiles

Dans le HTML, la note est encodée en **anglais** dans la classe CSS :  
`<p class="star-rating Three">` → 3 étoiles

Complétez le dictionnaire de conversion :

In [ ]:

etoiles_map = {
    'One'  : '★',
    'Two'  : '★★',
    'Three': '★★★',
    'Four' : '★★★★',
    'Five' : '★★★★★'
}


print(etoiles_map.get('Three', '?'))   # → ★★★
print(etoiles_map.get('Five',  '?'))   # → ★★★★★

★★★
★★★★★


## Étape 6 — Tester l'extraction sur un seul livre

> 💡 **Bonne pratique :** testez toujours sur **un seul élément** avant de boucler !

| Donnée | Comment l'extraire |
|--------|--------------------|
| Titre complet | `article.find('h3').find('a')['title']` |
| Prix | `article.find('p', class_='price_color').text.strip()` |
| Note (mot) | `article.find('p', class_='star-rating')['class'][1]` → ex: `'Three'` |
| Disponibilité | `article.find('p', class_='instock').text.strip()` |

In [ ]:
premier = articles[0]


titre       = premier.find('h3').find('a')['title']
prix        = premier.find('p', class_='price_color').text.strip()
note_classe = premier.find('p', class_='star-rating')['class'][1]
note        = etoiles_map.get(note_classe, '?')
dispo       = premier.find('p', class_='instock').text.strip()


print(f"Titre : {titre}")
print(f"Prix  : {prix}")
print(f"Note  : {note}")
print(f"Dispo : {dispo}")

Titre : A Light in the Attic
Prix  : Â£51.77
Note  : ★★★
Dispo : In stock


## Étape 7 — Boucler sur tous les livres

Répétez l'extraction pour **chaque article** et ajoutez un dictionnaire par livre dans la liste `livres`.

Pour la disponibilité : convertissez `'In stock'` → `'Oui'` et autre → `'Non'`  
**Indice :** `'Oui' if 'In stock' in dispo else 'Non'`

In [ ]:
livres = []

for article in articles:

    titre       = article.find('h3').find('a')['title']
    prix        = article.find('p', class_='price_color').text.strip()
    note_classe = article.find('p', class_='star-rating')['class'][1]
    note        = etoiles_map.get(note_classe, '?')
    dispo_text  = article.find('p', class_='instock').text.strip()


    livres.append({
        'titre'      : titre,
        'prix'       : prix,
        'note'       : note,
        'disponible' : 'Oui' if 'In stock' in dispo_text else 'Non'
    })

print(f"✅ {len(livres)} livres extraits")

✅ 20 livres extraits


## Étape 8 — Créer le DataFrame et afficher les résultats

In [ ]:


df = pd.DataFrame(livres)


print("Résultats :")
print(df.head(10).to_string())

Résultats :
                                                                                            titre     prix   note disponible
0                                                                            A Light in the Attic  Â£51.77    ★★★        Oui
1                                                                              Tipping the Velvet  Â£53.74      ★        Oui
2                                                                                      Soumission  Â£50.10      ★        Oui
3                                                                                   Sharp Objects  Â£47.82   ★★★★        Oui
4                                                           Sapiens: A Brief History of Humankind  Â£54.23  ★★★★★        Oui
5                                                                                 The Requiem Red  Â£22.65      ★        Oui
6                                              The Dirty Little Secrets of Getting Your Dream Job  Â£33.34   ★★★★

## Étape 9 — Sauvegarder en CSV

In [ ]:


df.to_csv('livres_scraping.csv', index=False, encoding='utf-8-sig')

print(f"✅ {len(livres)} livres sauvegardés dans livres_scraping.csv")

✅ 20 livres sauvegardés dans livres_scraping.csv


## Étape 10 — Statistiques *(Bonus)*

Le prix scraped ressemble à `'Â£51.77'`. Pour le convertir en nombre, il faut nettoyer les caractères parasites :
```python
df['prix'].str.replace('Â','').str.replace('£','').astype(float)
```

In [ ]:


print("📈 Statistiques :")


prix_numerique = df['prix'].str.replace('Â','').str.replace('£','').astype(float)

print(f"   Prix moyen : {prix_numerique.mean():.2f}£")



print(f"   Livres 5 étoiles : {len(df[df['note'] == '★★★★★'])}")


print(f"   En stock : {len(df[df['disponible'] == 'Oui'])}/{len(df)}")

📈 Statistiques :
   Prix moyen : 38.05£
   Livres 5 étoiles : 4
   En stock : 20/20


---
##  Challenge — Scraper plusieurs pages *(Avancé)*

Le site a 50 pages. L'URL de la page suivante suit ce format :  
`https://books.toscrape.com/catalogue/page-2.html`

Scrapez les **3 premières pages** et combinez les résultats dans un seul DataFrame.

In [ ]:

BASE_URL = "https://books.toscrape.com/catalogue/page-{}.html"
tous_livres = []


for num_page in range(1, 4):

    response = requests.get(url_page, timeout=10)
    soup = BeautifulSoup(response.text, "lxml")


    articles = soup.find_all("article", class_="product_pod")


    for article in articles:
        titre       = article.find('h3').find('a')['title']
        prix        = article.find('p', class_='price_color').text.strip()
        note_classe = article.find('p', class_='star-rating')['class'][1]
        note        = etoiles_map.get(note_classe, '?')
        dispo_text  = article.find('p', class_='instock').text.strip()

        tous_livres.append({
            'titre'      : titre,
            'prix'       : prix,
            'note'       : note,
            'disponible' : 'Oui' if 'In stock' in dispo_text else 'Non'
        })


df_complet = pd.DataFrame(tous_livres)
print(f"📚 Total : {len(df_complet)} livres sur 3 pages")



NameError: name 'url_page' is not defined